# gap停滞とattribution — 「どの分枝が双対境界を押し上げたか」を測る

gapが縮まらないとき、SCIPのログは「ノード数」と「暫定gap」しか教えてくれない。
`minlpkit.collectors.attribution` は分枝ごとに大域双対境界のスナップショットを取り、
**双対境界の増分をその直前の分枝(変数と型)に帰属**させることで「効いた分枝はどれか」を
定量化する。あわせて **停滞区間の自動検出**(平均より改善レートが遅い時間帯)も行う。

1. **何を見るか** — `NODEBRANCHED` ごとの双対境界スナップショットと `Δdual` の帰属方法
2. **可視化** — 双対境界推移 + 型別寄与の積み上げ + 停滞区間のシェード
3. **読み解き** — 停滞は「横ばい」でなく「平均より遅い」で検出する理由、`dual_stall` 診断との対応
4. **まとめ**

題材は **バッチ反応器スケジューリング**(`samples/others/scheduling_plant.py`)、
既知の難問(`.claude/skills/minlp-run/SKILL.md`: 300秒でgap72.5%まで停滞)。

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp
from minlpkit.collectors.attribution import (
    AttributionCollector, attribute_gains, gain_by_kind, gain_by_variable,
    detect_stalls, solve_and_attribute)

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
KIND_COLOR = {"spatial": C["s1"], "integer": C["s2"], "binary": "#e87ba4", "root": C["muted"]}
KIND_LABEL = {"spatial": "空間分枝(連続)", "integer": "整数分枝", "binary": "0-1分枝", "root": "根"}
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 何を見るか — 分枝ごとの双対境界スナップショットと帰属

`AttributionCollector` は `NODEBRANCHED` のたびに `(経過時間, ノード数, 大域双対境界,
分枝変数, kind)` を1行記録する。`BESTSOLFOUND` は別途 TTFF(最初の可行解までの秒)の計測に使う。

`attribute_gains` はレコード列を時刻順に並べ、**レコード i の分枝が i→i+1 の大域双対境界の
押し上げに寄与した**とみなして `Δdual = max(0, dual[i+1] - dual[i])` を計算する。近似的な
信用割当だが、「spatial(連続への空間分枝) と discrete(整数/0-1分枝) のどちらが双対境界を
押しているか」という傾向を掴むには十分。

In [2]:
col = AttributionCollector()
m = sp.build_model()
m.includeEventhdlr(col, "AttributionCollector", "attributes dual gains")
m.setParam("timing/clocktype", 2)
m.setParam("limits/time", 60)
m.hideOutput()
m.optimize()
raw = col.to_frame()
d = attribute_gains(raw)
print(f"分枝レコード数: {len(raw)}  求解時間: {m.getSolvingTime():.1f}s  "
     f"最終gap: {m.getGap():.1%}  TTFF: {col.first_sol_time:.2f}s")
d[["time", "nodes", "dual", "branch_var", "kind", "dual_gain"]].head(6)

分枝レコード数: 20154  求解時間: 60.0s  最終gap: 95.2%  TTFF: 0.00s


,time,nodes,dual,branch_var,kind,dual_gain
0,0.593659,1,52.128421,NaN,root,0.000000
1,0.603075,2,52.128421,t_x_J4_M2,binary,0.000000
2,0.632163,3,52.128421,t_n_J4,integer,0.000000
3,0.635350,4,52.128421,t_x_J6_M2,binary,0.000000
4,0.638755,5,52.128421,t_x_J5_M2,binary,0.000000
5,0.669989,6,52.128421,t_n_J2,integer,0.426041


## 2. 可視化 — 双対境界推移 + 型別寄与 + 停滞区間

`detect_stalls` は時間を等間隔グリッドに載せ、双対境界を累積最大で補間したうえで、
**セルごとの改善量が全体平均セル改善量の半分未満**の区間を「停滞」として抽出する
(「横ばい」で判定すると、実際は連続的に改善しているケースを見落とすため)。

In [3]:
stalls = detect_stalls(d)
print(f"検出された停滞区間: {len(stalls)}個")
for a, b in stalls:
    print(f"  {a:.1f}s - {b:.1f}s (幅{b-a:.1f}s)")

fig = go.Figure()
for a, b in stalls:
    fig.add_vrect(x0=a, x1=b, fillcolor=C["muted"], opacity=0.15, line_width=0)
dd = d.dropna(subset=["dual"])
fig.add_trace(go.Scatter(x=dd["time"], y=dd["dual"], mode="lines",
    line=dict(color=C["s2"], width=2, shape="hv"), name="大域双対境界"))
for kind in ["spatial", "integer", "binary"]:
    kd = d[(d["kind"] == kind) & (d["dual_gain"] > 0)]
    if kd.empty:
        continue
    fig.add_trace(go.Scatter(x=kd["time"], y=kd["dual"], mode="markers",
        marker=dict(color=KIND_COLOR[kind], size=6), name=f"改善: {KIND_LABEL[kind]}"))
fig.update_layout(
    title=dict(text="双対境界推移(灰帯=改善が平均の半分未満の停滞区間)",
              font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="経過時間[s]", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    yaxis=dict(title="大域双対境界", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=60, r=20, t=48, b=44), height=420,
    legend=dict(orientation="h", y=1.0, yanchor="bottom", x=1.0, xanchor="right", font=dict(size=11)))
show(fig)

検出された停滞区間: 1個
  44.4s - 48.9s (幅4.4s)


In [4]:
gk = gain_by_kind(d)
total = gk["dual_gain"].sum()
fig = go.Figure(go.Bar(
    x=gk["kind"].map(lambda k: KIND_LABEL.get(k, k)), y=gk["dual_gain"],
    marker=dict(color=[KIND_COLOR.get(k, C["muted"]) for k in gk["kind"]]),
    text=[f"{v/total:.0%}" for v in gk["dual_gain"]], textposition="outside"))
fig.update_layout(
    title=dict(text="双対境界改善の型別内訳(帰属合計)",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    yaxis=dict(title="Δdual 合計", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=60, r=20, t=44, b=40), height=300)
show(fig)

gv = gain_by_variable(d, top=10).iloc[::-1]
fig2 = go.Figure(go.Bar(x=gv["dual_gain"], y=gv["branch_var"], orientation="h",
    marker=dict(color=[KIND_COLOR.get(k, C["muted"]) for k in gv["kind"]]),
    text=[f"{v:.1f}" for v in gv["dual_gain"]], textposition="outside"))
fig2.update_layout(
    title=dict(text="双対境界改善への寄与が大きい変数トップ10",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="Δdual 合計", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=110, r=40, t=44, b=40), height=340)
show(fig2)
print(f"spatial寄与 = {gk.loc[gk['kind']=='spatial','dual_gain'].sum()/total:.1%}")

spatial寄与 = 62.8%


## 3. 読み解き — 停滞と `dual_stall` 診断ルールとの対応

`dual_stall` ルールは `n_stalls>=1` かつ `gap>=0.05` で発火し、
「有効不等式の追加・変数境界タイト化・Big-M排除」を推薦する。ただし推薦は
「症状はある」ことしか教えないため、実際に効くかは `mk.compare_variants` での
before/after 検証が必須(手法ガイドの各手法ページ参照)。

In [5]:
report = mk.analyze(sp.build_model, name="plant-attribution", time_limit=15)
print(report.summary())
for f in report.findings:
    if f["id"] in ("dual_stall", "weak_relaxation"):
        print(f"\n[{f['severity']}] {f['id']}: {f['symptom']}")
        print("  根拠:", f["evidence"])
        print("  手順:", f["recipe"])

[plant-attribution] 検出症状 4件:
  [serious] 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い) -> ボトルネック制約の区分線形近似・凸包再定式化・変数境界タイト化
  [warning] 双対境界の改善が停滞(gapが残る) -> 有効不等式の追加・変数境界タイト化・Big-M排除で緩和強化
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化
  [good] 制約-変数がブロック構造 + 少数の結合制約 -> ベンダーズ分解 / Dantzig-Wolfe分解(結合制約を主問題に)

[serious] weak_relaxation: 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い)
  根拠: 支配ボトルネック=energy(相対違反1.00)、空間分枝の双対寄与54%
  手順: 整数×連続の積は mk.linearize_product(m,y,x,...) で厳密線形化。非線形1変数は mk.pwl_sos2(m,x,brks,vals) で区分線形化。例: scheduling_plant(n·s)→improve_linearize.html, sos.html

[warning] dual_stall: 双対境界の改善が停滞(gapが残る)
  根拠: 停滞区間3個、最終gap136.8%
  手順: 有効不等式の追加・変数境界タイト化・Big-M排除で緩和を強化。効果は mk.compare_variants で検証。例: attribution.html


## まとめ

- attribution 収集器は「双対境界がいつ、何によって改善したか」を分枝単位で帰属させる、
  分枝限定法のブラックボックスを開ける道具。停滞区間の自動検出は「横ばい」でなく
  「平均より遅い」基準にすることで、連続的にゆっくり改善するケースも正しく拾う。
- `gain_by_kind` の内訳は `weak_relaxation`(空間分枝が支配的なら非凸緩和が律速)と
  `decomposable`/`dual_stall` の判定材料を兼ねる — 1つの収集器が複数の診断ルールを支える。
- 停滞区間や寄与変数が特定できても、それ自体は「打ち手」ではない。実際に効くかは
  [手法ガイド](../../playbook/index.md)の該当ページ(各ページからnotebookにリンク)で測るのが次の一手。

関連: [手法ガイド 0. 診断そのもの](../../playbook/00-diagnose.md) /
[notebook 1: McCormick+空間分枝木](01_mccormick_spatial_tree.ipynb) /
API [`minlpkit.collectors`](../../api/pipeline.md)。